# Chatterbox TTS on Colab GPU (remote backend)

Run this notebook on Google Colab with a **GPU runtime** (T4 is free; A100 is faster). It boots `travisvn/chatterbox-tts-api` on Colab's GPU and exposes it through your **stable ngrok domain**, so `CHATTERBOX_API_URL` in your local `.env` is set once and reused across every Colab session.

## One-time setup (do this before running cells)
1. **Runtime → Change runtime type → GPU.**
2. **Tools → Secrets** (🔑 in the left sidebar) → add two secrets, both with **"Notebook access"** enabled:
   - `NGROK_AUTHTOKEN` — your ngrok authtoken (from ngrok dashboard → *Your Authtoken*).
   - `NGROK_DOMAIN` — your claimed static domain, **bare hostname only**, e.g. `something-stable.ngrok-free.dev` (no `https://`, no path).

## Per-session usage
1. Run all cells. Cell 4 prints `Public URL: https://<your-NGROK_DOMAIN>`.
2. On your Mac, set `CHATTERBOX_API_URL=https://<your-NGROK_DOMAIN>` in `.env` once. Subsequent Colab sessions reuse the same URL — no `.env` edit needed.
   ```bash
   echo 'CHATTERBOX_API_URL=https://<your-NGROK_DOMAIN>' >> .env
   docker compose --profile cpu up -d --no-deps --force-recreate api
   ```
3. (Optional) Stop the local `chatterbox-cpu` container — it's no longer needed:
   ```bash
   docker compose --profile cpu stop chatterbox-cpu
   ```
4. Trigger TTS from the frontend at `http://localhost:8501`. Each segment now runs on Colab's GPU.

**Important:** Colab kills idle GPU sessions after ~90 minutes. Keep this tab open during long jobs.

## Why a Python 3.11 venv
Colab now ships Python 3.12, but `chatterbox-tts` pins `numpy<1.26` which has no 3.12 wheel and won't build from source on 3.12. Cell 2 provisions a Python 3.11 venv at `/content/venv311` just for the chatterbox subprocess — the notebook kernel stays on 3.12.


## 1. Verify GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    'CUDA is not available — Runtime → Change runtime type → GPU, then re-run this cell.'
)
print(f'CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0)}')

## 2. Install chatterbox-tts-api into a Python 3.11 venv

In [ ]:
!git clone --depth 1 https://github.com/travisvn/chatterbox-tts-api.git /content/chatterbox-tts-api

# Colab ships Python 3.12, but chatterbox-tts pins numpy<1.26 which has no
# 3.12 wheels. Provision a separate Python 3.11 venv just for the chatterbox
# subprocess; the kernel itself can stay on 3.12.
!pip install -q uv
!uv venv --python 3.11 /content/venv311

# `uv venv` does not install pip inside the venv (uv expects you to use
# `uv pip install --python <interp>`). All installs below target the venv
# interpreter directly via uv pip.
!uv pip install --python /content/venv311/bin/python git+https://github.com/travisvn/chatterbox-multilingual.git@exp
!uv pip install --python /content/venv311/bin/python -r /content/chatterbox-tts-api/requirements.txt

# Explicitly pin the runtime extras that requirements.txt sometimes drops
# in fresh venvs, plus `peft` which recent diffusers requires for
# LoRACompatibleLinear (without it the model init silently returns None
# and /v1/audio/speech 500s on first call).
!uv pip install --python /content/venv311/bin/python "uvicorn[standard]>=0.24.0" "fastapi>=0.104.0" "peft>=0.7.0"

# Comprehensive smoke test — fail loudly here if any of the imports main.py
# needs are missing, instead of letting cell 4 surface them later.
!/content/venv311/bin/python -c "import chatterbox, uvicorn, fastapi, peft, torch; print('all imports ok; torch=', torch.__version__, 'cuda=', torch.cuda.is_available())"


## 3. Install ngrok and load tunnel credentials

This notebook uses an ngrok static domain so the public URL is stable across sessions. Set `NGROK_AUTHTOKEN` and `NGROK_DOMAIN` in **Tools → Secrets** (🔑 in the left sidebar) before running. Both must have "Notebook access" enabled.

In [ ]:
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
NGROK_DOMAIN = (userdata.get('NGROK_DOMAIN') or '').strip()
# Strip any scheme/path the user pasted into the secret by mistake — we
# only want the bare hostname so we can prefix `https://` ourselves.
NGROK_DOMAIN = NGROK_DOMAIN.replace('https://', '').replace('http://', '').strip('/')

assert NGROK_AUTHTOKEN and NGROK_DOMAIN, (
    'Set NGROK_AUTHTOKEN and NGROK_DOMAIN in Colab Secrets (Tools → 🔑) before running this cell.'
)

# pyngrok auto-downloads the right ngrok binary for the current platform —
# no fragile curl/tar dance. Idempotent.
%pip install -q pyngrok
from pyngrok import conf
conf.get_default().auth_token = NGROK_AUTHTOKEN
print(f'pyngrok configured; tunnel domain: {NGROK_DOMAIN}')


## 4. Boot Chatterbox + tunnel

Both run as background subprocesses. The cell waits for chatterbox to become healthy, then for the tunnel URL, and prints both. If chatterbox crashes, the cell raises immediately with the last 50 lines of `tts.log`.

In [ ]:
import os, json, subprocess, time, pathlib

# ── Secrets (re-read so this cell works even if cell 3 was skipped) ─
from google.colab import userdata
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
NGROK_DOMAIN = (userdata.get('NGROK_DOMAIN') or '').strip()
NGROK_DOMAIN = NGROK_DOMAIN.replace('https://', '').replace('http://', '').strip('/')
assert NGROK_AUTHTOKEN and NGROK_DOMAIN, (
    'Set NGROK_AUTHTOKEN and NGROK_DOMAIN in Colab Secrets (Tools → 🔑) before running this cell.'
)

# ── Self-heal: ensure pyngrok is installed and configured ──────────
try:
    from pyngrok import ngrok, conf
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'pyngrok'], check=True)
    from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN

# ── Kill any stale ngrok holding the static domain from a prior run ─
ngrok.kill()
time.sleep(1)

# ── Chatterbox env ─────────────────────────────────────────────────
os.environ['DEVICE'] = 'cuda'
os.environ['DEFAULT_MODEL'] = 'multilingual'
os.environ['PORT'] = '8020'
os.environ['HOST'] = '0.0.0.0'

log_dir = pathlib.Path('/content/logs'); log_dir.mkdir(exist_ok=True)
tts_log_path = log_dir / 'tts.log'
tts_log = open(tts_log_path, 'wb')

VENV_PYTHON = '/content/venv311/bin/python'

# ── Boot chatterbox (skip if already serving on :8020) ─────────────
import urllib.request, urllib.error

def _model_ready():
    """Probe the actual synthesis endpoint, not just /health.

    /health returns 200 the moment FastAPI binds the port, but chatterbox
    loads its model asynchronously in the background — calls to
    /v1/audio/speech 500 until the model is fully loaded. This is the
    only reliable readiness signal.
    """
    try:
        req = urllib.request.Request(
            'http://127.0.0.1:8020/v1/audio/speech',
            data=json.dumps({'input': 'ok', 'response_format': 'wav'}).encode(),
            headers={'Content-Type': 'application/json'},
            method='POST',
        )
        with urllib.request.urlopen(req, timeout=120) as r:
            return r.status == 200
    except urllib.error.HTTPError:
        return False  # 500 while model is loading; keep polling
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        return False  # server not up yet; keep polling

if _model_ready():
    print('chatterbox-tts-api already healthy — reusing')
    tts = None
else:
    tts = subprocess.Popen(
        [VENV_PYTHON, 'main.py'],
        cwd='/content/chatterbox-tts-api',
        env=os.environ.copy(),
        stdout=tts_log, stderr=subprocess.STDOUT,
    )
    print('Booting chatterbox — waiting for model to finish loading '
          '(this can take 1–3 minutes on first cold run)...')
    deadline = time.time() + 900   # 15 min budget for cold model fetch + load
    healthy = False
    last_print = 0
    while time.time() < deadline:
        rc = tts.poll()
        if rc is not None:
            tail = pathlib.Path(tts_log_path).read_text(errors='replace').splitlines()[-50:]
            raise RuntimeError(
                f'chatterbox-tts-api exited with code {rc} before becoming healthy.\n'
                f'Last 50 lines of {tts_log_path}:\n' + '\n'.join(tail)
            )
        if _model_ready():
            print('chatterbox-tts-api is healthy (model loaded, /v1/audio/speech is 200)')
            healthy = True
            break
        # Light progress heartbeat every 30 s.
        if time.time() - last_print > 30:
            elapsed = int(time.time() - (deadline - 900))
            print(f'  ...still loading after {elapsed}s')
            last_print = time.time()
        time.sleep(10)
    if not healthy:
        raise RuntimeError(
            f'chatterbox-tts-api never produced a successful /v1/audio/speech '
            f'within 15 min; check {tts_log_path}'
        )

# ── Boot ngrok tunnel against the static domain ────────────────────
http_tunnel = ngrok.connect(8020, 'http', domain=NGROK_DOMAIN)
url = http_tunnel.public_url
if url.startswith('http://'):
    url = 'https://' + url[len('http://'):]

# Verify the tunnel actually answers a real synthesis request — not
# just /health. This guarantees the URL we hand the user is fully usable.
deadline = time.time() + 60
ready = False
while time.time() < deadline:
    try:
        req = urllib.request.Request(
            f'{url}/v1/audio/speech',
            data=json.dumps({'input': 'ok', 'response_format': 'wav'}).encode(),
            headers={'Content-Type': 'application/json'},
            method='POST',
        )
        with urllib.request.urlopen(req, timeout=60) as r:
            if r.status == 200:
                ready = True
                break
    except (urllib.error.URLError, urllib.error.HTTPError, ConnectionError, TimeoutError):
        time.sleep(3)

if not ready:
    raise RuntimeError(f'ngrok tunnel for {url} did not pass a real synthesis probe')

print('=' * 70)
print('Public URL:', url)
print('=' * 70)
print('On your Mac (only needs to be done once — this URL is stable):')
print(f"  echo 'CHATTERBOX_API_URL={url}' >> .env")
print('  docker compose --profile cpu up -d --no-deps --force-recreate api')


## 5. Smoke test

Confirm the public URL actually answers TTS calls before pointing the Mac at it.

In [ ]:
import urllib.request, json
req = urllib.request.Request(
    f'{url}/v1/audio/speech',
    data=json.dumps({'input': 'hola desde colab', 'response_format': 'wav'}).encode(),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with urllib.request.urlopen(req, timeout=120) as r:
    data = r.read()
print(f'WAV bytes: {len(data)}')

## 6. Keep alive

Run this last cell to block the kernel and stream the chatterbox log. As long as it's running, the tunnel stays up. Stop it with the square button when done.

In [ ]:
!tail -F /content/logs/tts.log